In [ ]:
import os

In [ ]:
# ALTERAR APENAS ESTAS DUAS VARIÁVEIS

area = "extremadura"
scenario_rf = "C6"


configs = {
    "C2": {
        "areas": ["centro"],
        "scenario_lr": "C1",
        "source": "icnf",
        "period": "1995-2024"
    },
    "C4": {
        "areas": ["centro"],
        "scenario_lr": "C3",
        "source": "icnf",
        "period": "2008-2024"
    },
    "C6": {
        "areas": ["centro", "extremadura"],
        "scenario_lr": "C5",
        "source": "effis",
        "period": "2008-2024"
    }
}


if scenario_rf not in configs:
    raise ValueError(
        f"Cenário RF inválido: {scenario_rf}"
    )


cfg = configs[scenario_rf]

if area not in cfg["areas"]:
    raise ValueError(
        f"O cenário {scenario_rf} não está disponível para {area}"
    )


scenario_lr = cfg["scenario_lr"]
source = cfg["source"]
ref_period = cfg["period"]
ref_file = ref_period.replace("-", "_")


rf_folder = (
    f"/code/data/results/{area}/"
    f"{scenario_rf}/lri_model"
)

validation_folder = os.path.join(
    rf_folder,
    "validation"
)

os.makedirs(
    validation_folder,
    exist_ok=True
)


base = f"/code/data/processed/{area}"


# Avaliação retrospetiva.
refs = {
    f"success_rate_{ref_period}": (
        f"{base}/area_ardida/{source}/raster_binary/"
        f"rst_ba_{ref_file}_bin.tif"
    )
}


# C4 e C6 são avaliados com as duas fontes em 2025.
if area == "centro" and scenario_rf in ["C4", "C6"]:
    refs.update({
        "prediction_rate_2025_icnf": (
            f"{base}/area_ardida/icnf/raster_binary/"
            "rst_ba_2025_bin.tif"
        ),
        "prediction_rate_2025_effis": (
            f"{base}/area_ardida/effis/raster_binary/"
            "rst_ba_2025_bin.tif"
        )
    })

else:
    refs[f"prediction_rate_2025_{source}"] = (
        f"{base}/area_ardida/{source}/raster_binary/"
        "rst_ba_2025_bin.tif"
    )


products = {
    "susc": (
        f"{rf_folder}/class/"
        "rf_lri_base_mean_prob_1_10models.tif"
    ),
    "peri": (
        f"{rf_folder}/class/"
        "res_perigosity_rf.tif"
    )
}


positive_value = 1

output_table = os.path.join(
    validation_folder,
    "rf_validation.xlsx"
)


print("Área:", area)
print("Cenário RF:", scenario_rf)
print("Cenário LR correspondente:", scenario_lr)
print("Fonte de treino:", source)
print("Período:", ref_period)
print("Referências:", refs)
print("Produtos:", products)

In [ ]:
import os
import sys

import pandas as pd

from glass.pys.oss import fprop
from glass.wt import obj_to_tbl

sys.path.append("/code/scripts")

from validation import pseudo_roc_blocks

In [ ]:
rows = []
curves = {}


for product, raster in products.items():
    raster_name = fprop(
        raster,
        "fn"
    )

    for evaluation, reference in refs.items():
        curve, auc_value, _ = pseudo_roc_blocks(
            ref=reference,
            perigo_rst=raster,
            posval=positive_value,
            otbl=None,
            block_size=1024
        )

        curve_name = (
            f"{scenario_rf}_{product}_{evaluation}"
        )

        curve_file = os.path.join(
            validation_folder,
            f"{curve_name}_curve.xlsx"
        )

        obj_to_tbl(
            curve,
            curve_file
        )

        curves[curve_name] = curve

        rows.append({
            "area": area,
            "scenario": scenario_rf,
            "method": "LR+RF",
            "training_source": source,
            "training_period": ref_period,
            "product": product,
            "raster": raster_name,
            "evaluation": evaluation,
            "reference": reference,
            "auc": auc_value,
            "curve_file": curve_file
        })

        print(
            scenario_rf,
            product,
            evaluation,
            round(auc_value, 6)
        )


validation_results = pd.DataFrame(
    rows
)

obj_to_tbl(
    validation_results,
    output_table
)

validation_results

In [ ]:
curve_name = (
    f"{scenario_rf}_peri_"
    f"prediction_rate_2025_{source}"
)

curves[curve_name].plot.scatter(
    x="tarearatio",
    y="tfireration"
)